In [ ]:
import pandas as pd

link_stream = pd.read_csv('data/link_stream.txt', delimiter=' ', names=['source', 'destination', 'timestamp'])

In [ ]:
import pandas as pd


nodes = pd.concat([link_stream['source'], link_stream['destination']]).unique()

node2id = {node: idx for idx, node in enumerate(nodes)}

link_stream['source'] = link_stream['source'].map(node2id)
link_stream['destination'] = link_stream['destination'].map(node2id)
link_stream['idx'] = range(len(link_stream))

In [ ]:
link_stream.to_csv('data/link_stream.csv', index=False)

In [ ]:
import pandas as pd

link_stream = pd.read_csv('data/link_stream.csv')

In [3]:
import torch
import pandas as pd

class OfflineStateManager: 
    def __init__(self, num_nodes, num_communities, device='cpu'):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device
        # Static parameters
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        # Initialize dynamic states
        self.node_lifespans = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0  # total number of links
        self.T_context = None
        self.T_buffer = torch.zeros(num_nodes, num_communities, device=device)
        self.last_seen = torch.full((num_nodes,), -1.0, device=device)
        self.last_prob = torch.ones(num_nodes, num_communities, device=device) / num_communities

    def pre_scan_data(self, link_stream):
        self.m = float(len(link_stream))

        all_nodes = pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True)
        all_nodes_t = torch.tensor(all_nodes.values, dtype=torch.long, device=self.device)
        self.global_degree = torch.bincount(all_nodes_t, minlength=self.num_nodes).float()

        t_max = link_stream['timestamp'].max()
        t_min = link_stream['timestamp'].min()
        total_duration = t_max - t_min

        print(f"T_duration = {total_duration}")
        self.total_duration = float(total_duration) if total_duration > 0 else 1.0
        print(f"Total links: {int(self.m)}")

        # compute node lifespans
        sources = link_stream[['source', 'timestamp']].rename(columns={'source': 'node'})
        destinations = link_stream[['destination', 'timestamp']].rename(columns={'destination': 'node'})
        all_events = pd.concat([sources, destinations], ignore_index=True)
        
        node_stats = all_events.groupby('node')['timestamp'].agg(['min', 'max'])
        
        epsilon = 1.0 
        lifespans = (node_stats['max'] - node_stats['min']).clip(lower=epsilon)
        
        # 填充 Tensor
        active_ids = torch.tensor(node_stats.index.values, device=self.device, dtype=torch.long)
        active_vals = torch.tensor(lifespans.values, device=self.device, dtype=torch.float32)
        
        self.node_lifespans.fill_(epsilon)
        self.node_lifespans[active_ids] = active_vals
        print(f"Max Lifespan: {self.node_lifespans.max().item()}")


    def init_context(self):
        """
        [Step 1] 在 pre_scan 后，训练开始前调用。
        使用生命周期初始化 T_Context。
        """
        # 初始化假设：节点把它的生命周期平均分配给所有社区
        # Shape: [N] -> [N, 1] -> [N, K]
        avg_duration = self.node_lifespans.unsqueeze(1) / self.num_communities
        self.T_context = avg_duration.repeat(1, self.num_communities)
        
        self.T_buffer.zero_()
        self.last_seen.fill_(-1.0) # 保持是一维
        self.last_prob.fill_(1.0 / self.num_communities)
    
    def on_epoch_start(self):
        """每个 Epoch 开始时调用：清空累加器，重置时间"""
        self.T_buffer.zero_()
        self.last_seen.fill_(-1.0)
        # 注意：Last_Prob 不重置，继承上一轮学到的先验分布

    def get_context_for_loss(self, node_ids):
        """
        [Read] 在计算 Loss 时调用
        """
        # 1. 获取 T_context 所在的设备 (可能是 CPU)
        target_device = self.T_context.device
        
        # 2. 将索引 (node_ids) 挪到同一个设备上
        # 如果 node_ids 已经在该设备，这步操作是零开销的
        ids_on_target = node_ids.to(target_device)
        
        # 3. 进行索引取值
        values = self.T_context[ids_on_target]
        
        # 4. 将取出的结果挪回 node_ids 原来的设备 (通常是 GPU，为了后续算 Loss)
        return values.to(node_ids.device)
    

    def get_last_prob(self, node_ids):
        """
        安全读取上一次的概率分布，自动处理 CPU/GPU 索引冲突
        """
        # 1. 获取存储位置 (CPU)
        target_device = self.last_prob.device 
        # 2. 搬运索引
        ids_on_target = node_ids.to(target_device)
        # 3. 取值
        values = self.last_prob[ids_on_target]
        # 4. 搬回原设备 (GPU)
        return values.to(node_ids.device)

    # [新增] 安全读取 global_degree
    def get_node_degree(self, node_ids):
        """
        安全读取节点度数，自动处理 CPU/GPU 索引冲突
        """
        target_device = self.global_degree.device
        ids_on_target = node_ids.to(target_device)
        values = self.global_degree[ids_on_target]
        return values.to(node_ids.device)
    

    def update_step(self, node_ids, new_probs, current_time):
        """
        [Write] 在 Optimizer.step() 之后调用
        逻辑：Sample-and-Hold 差分累积
        """
        node_ids = node_ids.to(self.device)
        new_probs = new_probs.detach().to(self.device)
        current_time = current_time.to(self.device)
        
        # 1. 获取上次状态
        prev_times = self.last_seen[node_ids]
        prev_probs = self.last_prob[node_ids]
        
        # 2. 识别非首次出现的节点
        # 只有之前出现过 (prev_time != -1)，才有"空窗期"需要填补
        mask = (prev_times != -1)
        
        if mask.any():
            valid_nodes = node_ids[mask]
            
            # 计算时间差 delta_t
            delta_t = current_time[mask] - prev_times[mask]
            # 确保时间非负 (防止数据乱序)
            delta_t = delta_t.clamp(min=0)
            
            # 计算时长增量: 旧分布 * 时间差
            # [M, 1] * [M, K]
            duration_inc = delta_t.unsqueeze(1) * prev_probs[mask]
            
            # 累加到 Buffer
            # 使用 index_add_ 处理 batch 内可能的重复节点
            self.T_buffer.index_add_(0, valid_nodes, duration_inc)

        # 3. 无论是否首次出现，都更新状态为当前值
        self.last_seen[node_ids] = current_time
        self.last_prob[node_ids] = new_probs

    def on_epoch_end(self):
        self.T_context = self.T_buffer.clone()
        self.T_buffer.zero_()
        # 重置时间
        self.last_seen.fill_(-1.0)

In [ ]:
import torch
import pandas as pd

class OnlineStateManager: 
    def __init__(self, num_nodes, num_communities, device='cpu'):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device
        # Static parameters
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        # Initialize dynamic states
        self.node_lifespans = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0  # total number of links
        self.T_context = None
        self.last_seen = torch.full((num_nodes,), -1.0, device=device)
        self.last_prob = torch.ones(num_nodes, num_communities, device=device) / num_communities


    def init_context(self):
        """
        [Step 1] 在 pre_scan 后，训练开始前调用。
        使用生命周期初始化 T_Context。
        """
        # 初始化假设：节点把它的生命周期平均分配给所有社区
        # Shape: [N] -> [N, 1] -> [N, K]
        avg_duration = self.node_lifespans.unsqueeze(1) / self.num_communities
        self.T_context = avg_duration.repeat(1, self.num_communities)
        self.last_seen.fill_(-1.0) # 保持是一维
        self.last_prob.fill_(1.0 / self.num_communities)
    

    def get_context_for_loss(self, node_ids):
        """
        [Read] 在计算 Loss 时调用
        """
        # 1. 获取 T_context 所在的设备 (可能是 CPU)
        target_device = self.T_context.device
        
        # 2. 将索引 (node_ids) 挪到同一个设备上
        # 如果 node_ids 已经在该设备，这步操作是零开销的
        ids_on_target = node_ids.to(target_device)
        
        # 3. 进行索引取值
        values = self.T_context[ids_on_target]
        
        # 4. 将取出的结果挪回 node_ids 原来的设备 (通常是 GPU，为了后续算 Loss)
        return values.to(node_ids.device)
    

    def get_last_prob(self, node_ids):
        """
        安全读取上一次的概率分布，自动处理 CPU/GPU 索引冲突
        """
        # 1. 获取存储位置 (CPU)
        target_device = self.last_prob.device 
        # 2. 搬运索引
        ids_on_target = node_ids.to(target_device)
        # 3. 取值
        values = self.last_prob[ids_on_target]
        # 4. 搬回原设备 (GPU)
        return values.to(node_ids.device)

    # [新增] 安全读取 global_degree
    def get_node_degree(self, node_ids):
        """
        安全读取节点度数，自动处理 CPU/GPU 索引冲突
        """
        target_device = self.global_degree.device
        ids_on_target = node_ids.to(target_device)
        values = self.global_degree[ids_on_target]
        return values.to(node_ids.device)
    

    def update_step(self, node_ids, new_probs, current_time):
        """
        [Write] 在 Optimizer.step() 之后调用
        逻辑：Sample-and-Hold 差分累积
        """
        node_ids = node_ids.to(self.device)
        new_probs = new_probs.detach().to(self.device)
        current_time = current_time.to(self.device)
        
        # 1. 获取上次状态
        prev_times = self.last_seen[node_ids]
        prev_probs = self.last_prob[node_ids]
        
        # 2. 识别非首次出现的节点
        # 只有之前出现过 (prev_time != -1)，才有"空窗期"需要填补
        mask = (prev_times != -1)
        
        if mask.any():
            valid_nodes = node_ids[mask]
            
            # 计算时间差 delta_t
            delta_t = current_time[mask] - prev_times[mask]
            # 确保时间非负 (防止数据乱序)
            delta_t = delta_t.clamp(min=0)
            
            # 计算时长增量: 旧分布 * 时间差
            # [M, 1] * [M, K]
            duration_inc = delta_t.unsqueeze(1) * prev_probs[mask]
            
            # 累加到 Buffer
            # 使用 index_add_ 处理 batch 内可能的重复节点
            self.T_context.index_add_(0, valid_nodes, duration_inc)

        # 3. 无论是否首次出现，都更新状态为当前值
        self.last_seen[node_ids] = current_time
        self.last_prob[node_ids] = new_probs

In [4]:
import torch

class GlobalDegreeSampler:
    def __init__(self, global_degrees):
        """
        初始化采样器
        global_degrees: Tensor [N], 也就是 state_mgr.global_degree
        """
        self.num_nodes = len(global_degrees)
        self.degrees = global_degrees.float()
        self.weights = self.degrees

    def sample(self, num_samples):
        """
        采样 num_samples 个负样本节点 ID
        返回: Tensor [num_samples]
        """
        # torch.multinomial 实现了加权无放回/有放回采样
        # replacement=True: 允许重复采样（在大图中通常没问题且更快）
        neg_samples = torch.multinomial(self.weights, num_samples, replacement=True)
        return neg_samples

In [6]:
node_set = set(pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True))
num_nodes = len(node_set)
print(f'num_nodes = {num_nodes} ')
num_comms = 5
state_mgr = OfflineStateManager(num_nodes, num_comms, device='cpu')
state_mgr.pre_scan_data(link_stream)
state_mgr.init_context()

num_nodes = 986 
T_duration = 69459254
Total links: 332334
Max Lifespan: 69440728.0


In [7]:
global_sampler = GlobalDegreeSampler(state_mgr.global_degree)

In [8]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class LinkStreamDataset(Dataset):
    def __init__(self, df):
        """
        df: 包含 u, v, t 的 DataFrame。
        """
        # 1. 存数据 (Numpy 格式，极快)
        self.src = df['source'].values.astype(np.int64)
        self.dst = df['destination'].values.astype(np.int64)
        self.times = df['timestamp' ].values.astype(np.float32)
        self.edge_idxs = df.index.values.astype(np.int64)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.dst[idx], self.times[idx], self.edge_idxs[idx]

def tgn_collate_fn(batch):
    """
    自定义整理函数：把 [(u1,v1,t1,i1), (u2,v2,t2,i2)] 变成 4 个 Tensor
    """
    # zip(*batch) 会把 list of tuples 转置为 tuple of lists
    src, dst, t, edge_idx = zip(*batch)
    
    return (
        torch.tensor(src, dtype=torch.long),
        torch.tensor(dst, dtype=torch.long),
        torch.tensor(t, dtype=torch.float32),
        torch.tensor(edge_idx, dtype=torch.long)
    )

dataset = LinkStreamDataset(link_stream)
dataloader = DataLoader(dataset, batch_size=500, shuffle=False, collate_fn=tgn_collate_fn)

In [9]:
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path

from tgn.evaluation.evaluation import eval_edge_prediction
from tgn.model.my_tgn import TGN
from tgn.utils.utils import EarlyStopMonitor, RandEdgeSampler, get_neighbor_finder
from tgn.utils.my_data import get_data, compute_time_statistics

NUM_EPOCHS = 3
BATCH_SIZE = 100
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'

prefix = 'max_lmod'
data='link_stream'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


In [10]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'link_stream'

node_feat, edge_feat, full_data = get_data(data)
max_idx = max(full_data.unique_nodes)

cannot find (./data/link_stream.npy), use zero-vector (dim=16)...
cannot find node feature: ./data/link_stream_node.npy), use zero vector(dim=16)...
The dataset has 332334 interactions, involving 986 different nodes


In [11]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [12]:
from tgn.utils.my_data import get_data, compute_time_statistics
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= \
  compute_time_statistics(
      full_data.sources, 
      full_data.destinations, 
      full_data.timestamps
  )


In [15]:
from tgn.model.my_tgn import TGN
importlib.reload(sys.modules['tgn.model.my_tgn'])
from pathlib import Path


NUM_EPOCHS = 5
BATCH_SIZE = 100
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'


aggregator = 'last'
memory_update_at_end = True
embedding_module = 'graph_attention'
message_function = 'mlp'

prefix = 'email'
n_runs = 2
for i in range(n_runs):
    results_path = "results/{}_community_{}.pkl".format(prefix, i) if i > 0 else "results/{}.pkl".format(prefix)
    Path("results/").mkdir(parents=True, exist_ok=True)

    tgn = TGN(
        neighbor_finder=ngh_finder,
        node_features=node_feat,
        edge_features=edge_feat,
        device=device,
        n_layers=NUM_LAYER,
        n_heads=NUM_HEADS,
        dropout=DROP_OUT,
        use_memory=USE_MEMORY,
        message_dimension=MESSAGE_DIM,
        memory_dimension=MEMORY_DIM,
        n_neighbors=NUM_NEIGHBORS,
        mean_time_shift=mean_time_shift,
        std_time_shift=std_time_shift,
        use_destination_embedding_in_message=True,
        use_source_embedding_in_message=True,

        memory_update_at_start=not memory_update_at_end,
        embedding_module_type=embedding_module,
        message_function=message_function,
        aggregator_type=aggregator, 

    ).to(device)

    optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

In [16]:
import math 
import logging
import time


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 3324


In [ ]:
# 确保你已经有了 train_loader
# train_loader = DataLoader(dataset, batch_size=..., collate_fn=tgn_collate_fn, shuffle=False)
from tgn.model.loss import criterion_l_modularity
num_neg = 5 # 每个正样本配 5 个负样本

for epoch in range(NUM_EPOCHS):
    state_mgr.on_epoch_start() # 清空 Buffer
    
    # TGN 内存重置 (每个 Epoch 开始时，把记忆抹去，从头演化)
    if tgn.use_memory:
        tgn.memory.__init_memory__()

    tgn.set_neighbor_finder(ngh_finder) # 训练用训练集的邻居查找器
    tgn.train()

    logger.info(f'Starting epoch {epoch}')

    for batch_idx, batch in enumerate(dataloader):
        # 1. 解包数据
        src, dst, t, edge_idxs = batch
        src = src.to(device)
        dst = dst.to(device)
        t = t.to(device)
        # edge_idxs 是为了 TGN 查找边特征用的
        
        optimizer.zero_grad()
        
        # ==========================================
        # A. 负采样 (Global Degree Based)
        # ==========================================
        # 采样结果: [Batch_Size * Num_Neg]
        neg_nodes = global_sampler.sample(len(src) * num_neg).to(device)
        
        # ==========================================
        # B. 前向传播 (Forward)
        # ==========================================
        # 1. 正样本 Embedding & Prob
        # 使用修改后的通用接口
        z_src = tgn.compute_temporal_embeddings(src, t)
        z_dst = tgn.compute_temporal_embeddings(dst, t)
        p_src = tgn.community_projector(z_src) # [B, K]
        p_dst = tgn.community_projector(z_dst) # [B, K]



        # ==========================================
        # C. 准备 Loss 所需的 Context (从 StateMgr 读取)
        # ==========================================
        # [关键] 必须 detach! 这些是历史常量，不参与梯度计算
        
        # 1. 历史时长 T (Context)
        T_src_ctx = state_mgr.get_context_for_loss(src).detach()       # [B, K]
        T_neg_ctx = state_mgr.get_context_for_loss(neg_nodes).detach() # [B*S, K]
        T_neg_ctx = T_neg_ctx.view(len(src), num_neg, -1)              # [B, S, K]
        
        # 2. 上一次的概率 (用于 Switch Cost)
        last_p_src = state_mgr.get_last_prob(src).detach()
        
        # 3. 源节点的度数 (用于 EMM 系数归一化)
        # 从 global_degree 中查表
        k_src = state_mgr.get_node_degree(src).detach() # [B]
        
        # ==========================================
        # D. 计算 Loss
        # ==========================================
        loss, l_obs, l_null, l_csc = criterion_l_modularity(
            p_src, p_dst, 
            T_src_ctx, T_neg_ctx,
            state_mgr.total_duration,
            last_p_src,
            omega=1.0 # Switch Cost 权重
        )
        
        # ==========================================
        # E. 反向传播 (Backward)
        # ==========================================
        loss.backward()
        optimizer.step()
        
        # ========================================== 
        # F. 更新 TGN Memory (仅正样本)
        # ==========================================
        if tgn.use_memory:
            with torch.no_grad():
                # TGN 需要"看到"刚才发生的真实交互 (src, dst, t) 来更新 GRU
                # 我们复用刚才算好的 z_src, z_dst 作为最新的 message
                uniq_s, msg_s = tgn.get_raw_messages(src, z_src, dst, z_dst, t, edge_idxs)
                uniq_d, msg_d = tgn.get_raw_messages(dst, z_dst, src, z_src, t, edge_idxs)
                
                # 存入消息队列
                tgn.memory.store_raw_messages(uniq_s, msg_s)
                tgn.memory.store_raw_messages(uniq_d, msg_d)
                
                # [关键] Detach Memory! 
                # 防止梯度流向无限久远的历史 (BPTT Truncation)
                tgn.memory.detach_memory()
        
        # ==========================================
        # G. 更新 State Manager (写新账)
        # ==========================================
        with torch.no_grad():
            # 使用 Sample-and-Hold 逻辑，把刚才算出的 p_src 累加到本轮账本 T_Buffer 中
            state_mgr.update_step(src, p_src, t)
            state_mgr.update_step(dst, p_dst, t)
            
            # 注意：负样本不更新状态，因为它们没有发生真实交互
            
    # Batch 循环结束
    # ==========================================
    
    # 交换 Buffer (本轮算好的 T_Buffer 变成下一轮的 T_Context)
    state_mgr.on_epoch_end()
    logger.info(f"Epoch {epoch} Loss: {loss.item():.4f}")

INFO:root:Starting epoch 0


Global T (每个节点在各社区的累计计数):
 tensor([[ 0., 17.],
        [ 0., 17.],
        [11.,  6.],
        [20.,  0.],
        [15.,  0.]])

--- Input Shapes ---
p_src: torch.Size([18, 2])
p_neg: torch.Size([18, 10, 2])
T_src: torch.Size([18, 2])

--- Test Results ---
Observed Loss (连边重合度): -1.0000 (越接近 -1 越好)
Null Loss (期望惩罚):       0.1564 (越小越好)
Switch Cost (切换惩罚):     0.1667  (越小越好)
Total Loss:                 -0.5103

[验证] 真实数据中同社区边比例: 18/18 = 1.00
[验证] 模型计算的 Obs Loss: -1.0000
>> Obs Loss 计算合理！(考虑到部分节点无标签使用了均匀分布)
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.

INFO:root:Epoch 0 Loss: -0.9997


torch.cuda.FloatTensor
torch.cuda.FloatTensor


In [ ]:
import pandas as pd
import torch
import numpy as np
import os

def export_community_structure(tgn, state_mgr, dataloader, device, output_path="results/community_structure.csv"):
    """
    执行推理，并将结果导出为 CSV
    格式: u, v, timestamp, u_comm, v_comm
    """
    print(f"开始导出社区结构到: {output_path}")
    
    # 1. 切换到评估模式
    tgn.eval()
    
    # 2. (可选) 重置状态 - 如果你希望输出的是基于最终训练好的模型的"完美回放"
    # 如果你想保留训练结束时的状态，注释掉下面两行
    state_mgr.on_epoch_start() 
    if tgn.use_memory:
        tgn.memory.__init_memory__()
        
    # 结果列表
    results = []
    
    # 3. 不计算梯度，节省显存
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            # 解包
            src, dst, t, edge_idxs = batch
            src = src.to(device)
            dst = dst.to(device)
            t = t.to(device)
            
            # A. 前向传播 (计算 Embedding)
            z_src = tgn.compute_temporal_embeddings(src, t)
            z_dst = tgn.compute_temporal_embeddings(dst, t)
            
            # B. 计算社区概率 (Probabilities)
            p_src = tgn.community_projector(z_src) # [B, K]
            p_dst = tgn.community_projector(z_dst) # [B, K]
            
            # C. 转换为社区 ID (Argmax)
            # dim=1 表示在社区维度上找最大值索引
            comm_src = torch.argmax(p_src, dim=1).cpu().numpy()
            comm_dst = torch.argmax(p_dst, dim=1).cpu().numpy()
            
            # D. 转回 CPU 以便存储
            src_cpu = src.cpu().numpy()
            dst_cpu = dst.cpu().numpy()
            t_cpu = t.cpu().numpy()
            
            # E. 收集结果
            for i in range(len(src)):
                results.append({
                    "u": src_cpu[i],
                    "v": dst_cpu[i],
                    "timestamp": t_cpu[i],
                    "u_comm": comm_src[i],
                    "v_comm": comm_dst[i]
                })
            
            # F. 别忘了更新 Memory！(推理时也需要更新记忆，否则后续的时间步无法利用之前的交互信息)
            if tgn.use_memory:
                uniq_s, msg_s = tgn.get_raw_messages(src, z_src, dst, z_dst, t, edge_idxs)
                uniq_d, msg_d = tgn.get_raw_messages(dst, z_dst, src, z_src, t, edge_idxs)
                tgn.memory.store_raw_messages(uniq_s, msg_s)
                tgn.memory.store_raw_messages(uniq_d, msg_d)
                tgn.memory.detach_memory() # 虽然是 no_grad，但加上保险
                
            # (可选) 打印进度
            if batch_idx % 100 == 0:
                print(f"Processing batch {batch_idx}...")

    # 4. 写入 CSV
    df = pd.DataFrame(results)
    
    # 确保目录存在
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    df.to_csv(output_path, index=False)
    print("导出完成！")
    return df

export_community_structure(tgn, state_mgr, dataloader, device, "results/final_communities.csv")

开始导出社区结构到: results/final_communities.csv
torch.cuda.FloatTensor
torch.cuda.FloatTensor
Processing batch 0...
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatTensor
torch.cuda.FloatT

,u,v,timestamp,u_comm,v_comm
0,0,824,0.0,2,2
1,1,216,2797.0,2,2
2,1,72,3304.0,2,2
3,2,152,4523.0,2,2
4,2,332,7926.0,2,2
...,...,...,...,...,...
332329,357,790,45401816.0,2,2
332330,152,208,45402440.0,2,2
332331,152,208,45403708.0,2,2
332332,342,208,45404904.0,2,2


: 